# 🤖 Fase 4 — Modelado · PANEL 2 (Predictivo)
### Trabajo Final · Minería de Datos · UNMSM-FISI · 2026-I

**Pregunta:** ¿este accidente de trabajo dejará al trabajador con una **secuela permanente**?

---
### Lo que exige el profe en este panel
| Requisito | Cómo lo cumplimos |
|---|---|
| **5 algoritmos** comparados | Log. Regression · Árbol · Random Forest · XGBoost · KNN |
| Matriz de confusión | ✅ para todos |
| Precisión, Recall, F1, ROC-AUC | ✅ tabla comparativa |
| **SHAP** (summary o force plot) | ✅ ambos |
| Justificación escrita del modelo elegido | ✅ al final |

### Las 3 salvaguardas del proyecto
1. **Anti-leakage de variables** — se excluyen `NATURALEZA_DE_LA_LESION` y `PARTE_DEL_CUERPO_LESIONADA` (son la *consecuencia*, no la causa).
2. **Anti-leakage temporal** — *split* por año (entrenar con el pasado, probar con el futuro) y las tasas de riesgo se calculan **solo con el train**.
3. **Anti-*concept drift*** — se usa únicamente el periodo **2018–2022** (ver abajo).


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

GRANATE, DORADO, AZUL, GRIS, VERDE = "#7a1128", "#d4a72c", "#3b6ea5", "#6b6b6b", "#2e7d5b"
sns.set_theme(style="whitegrid", palette=[GRANATE, DORADO, AZUL, VERDE, GRIS])
plt.rcParams.update({"figure.dpi":110, "axes.titlesize":12, "axes.titleweight":"bold",
                     "axes.titlecolor":GRANATE, "font.size":10, "legend.frameon":False})

df = pd.read_csv("../data set/limpio.csv")
print(f"Dataset limpio: {df.shape[0]:,} filas")


---
## ⚠️ Hallazgo crítico: *CONCEPT DRIFT* (cambio de criterio del MTPE)

Antes de modelar, revisamos si el **significado del target se mantiene estable en el tiempo**.
Si el organismo cambió su forma de clasificar, entrenar con un régimen y evaluar con otro
produciría un modelo que **falla sin culpa propia**.


In [ ]:
tasa_anual = df.groupby("ANIOS")["PERMANENTE"].agg(["size","mean"])
tasa_anual.columns = ["n_accidentes", "tasa_permanente"]
tasa_anual["tasa_permanente"] *= 100

fig, ax = plt.subplots(figsize=(11, 3.8))
colores = [VERDE if 2018 <= a <= 2022 else GRIS for a in tasa_anual.index]
ax.bar(tasa_anual.index, tasa_anual["tasa_permanente"], color=colores)
ax.set_title("% de accidentes con secuela PERMANENTE, por año")
ax.set_ylabel("% permanentes"); ax.set_xlabel("")
ax.axvspan(2017.5, 2022.5, color=VERDE, alpha=0.08)
ax.text(2020, tasa_anual["tasa_permanente"].max()*0.85,
        "régimen ESTABLE\n(2018–2022)", ha="center", color=VERDE, fontsize=9, weight="bold")
for a, v in zip(tasa_anual.index, tasa_anual["tasa_permanente"]):
    ax.text(a, v + 0.4, f"{v:.0f}%", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

display(tasa_anual.round(1))


### Diagnóstico

| Periodo | % permanentes | Interpretación |
|---|---|---|
| 2012–2017 | 0.5 % – 9 % | Registro incipiente / inconsistente |
| **2018–2022** | **17 % – 21 %** | ✅ **Régimen estable** |
| 2023–2024 | 0.3 % | ❌ El MTPE **cambió el criterio**: aparece la categoría *"PARCIAL TEMPORAL"* (13 %) que **no existía antes**, y *"PARCIAL PERMANENTE"* se desploma de 20 % a 0.2 % |

> **No es que los accidentes se volvieran menos graves: cambió la etiqueta.**
> Esto es ***concept drift***. Entrenar con 2018–2022 y evaluar en 2023 daría un recall pésimo
> **por culpa de los datos, no del modelo**.
>
> **Decisión:** restringimos el modelado a **2018–2022**. La exclusión se documenta como hallazgo
> de calidad (dimensiones **Consistencia** y **Oportunidad**).
> *(La serie completa 2012–2024 SÍ se conserva para el Panel 3, que solo cuenta accidentes por mes.)*


In [ ]:
df = df[(df["ANIOS"] >= 2018) & (df["ANIOS"] <= 2022)].copy()
print(f"Periodo de modelado: 2018–2022 → {df.shape[0]:,} accidentes")
print(f"Tasa de permanentes: {df['PERMANENTE'].mean()*100:.1f}%  "
      f"(ratio 1:{(1-df['PERMANENTE'].mean())/df['PERMANENTE'].mean():.1f})")


---
## 1️⃣ Split TEMPORAL (no aleatorio)

> **¿Por qué temporal y no `train_test_split` aleatorio?**
> Un split aleatorio mezclaría accidentes de 2022 en el entrenamiento y de 2018 en el test:
> el modelo estaría **viendo el futuro**. El split temporal simula la realidad —
> *"entreno con lo que ya pasó y predigo lo que viene"*.

- **Train:** 2018–2021
- **Test:** 2022


In [ ]:
train = df[df["ANIOS"] <= 2021].copy()
test  = df[df["ANIOS"] == 2022].copy()

print(f"TRAIN (2018–2021): {len(train):>7,} accidentes | {train['PERMANENTE'].mean()*100:.1f}% permanentes")
print(f"TEST  (2022)     : {len(test):>7,} accidentes | {test['PERMANENTE'].mean()*100:.1f}% permanentes")


---
## 2️⃣ Feature Engineering **dentro del split** (anti-leakage)

Las *tasas históricas de riesgo* se calculan **SOLO con el train** y luego se aplican al test.
Calcularlas con todo el dataset sería *target leakage*: el test le estaría "soplando" su respuesta al train.


In [ ]:
CAT = ["REGION","ACTIVIDAD_ECONOMICA","SEXO","CATEGORIA_OCUPACIONAL",
       "FORMA_DEL_ACCIDENTE_G","AGENTE_CAUSANTE_G","ESTACION"]

# --- Target encoding aprendido SOLO del train ---
tasa_global = train["PERMANENTE"].mean()
for col, nombre in [("ACTIVIDAD_ECONOMICA","TASA_SECTOR"),
                    ("REGION","TASA_REGION"),
                    ("FORMA_DEL_ACCIDENTE_G","TASA_FORMA")]:
    mapa = train.groupby(col)["PERMANENTE"].mean()          # ← aprendido del TRAIN
    train[nombre] = train[col].map(mapa)
    test[nombre]  = test[col].map(mapa).fillna(tasa_global)  # categorías nuevas → tasa global

NUM = ["MES_N","TRIMESTRE","ES_FIN_DE_ANIO","TASA_SECTOR","TASA_REGION","TASA_FORMA"]

# One-Hot ajustado en train y alineado en test
X_train = pd.get_dummies(train[CAT + NUM], columns=CAT, drop_first=True, dtype=int)
X_test  = pd.get_dummies(test[CAT + NUM],  columns=CAT, drop_first=True, dtype=int)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)   # misma estructura

y_train, y_test = train["PERMANENTE"], test["PERMANENTE"]
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


In [ ]:
# Z-Score ajustado SOLO en train (el scaler NO ve el test)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train[NUM])
X_train[NUM] = scaler.transform(X_train[NUM])
X_test[NUM]  = scaler.transform(X_test[NUM])
print("Z-Score aplicado ✓  (scaler ajustado solo con train)")


---
## 3️⃣ SMOTE — **solo sobre el train**

> **Regla de oro:** SMOTE **NUNCA** se aplica al test. Evaluar con datos sintéticos
> daría métricas falsamente buenas. El test debe conservar la proporción **real** del mundo.


In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (yy, t) in zip(axes, [(y_train, "TRAIN original"), (y_train_sm, "TRAIN tras SMOTE")]):
    c = yy.value_counts().sort_index()
    ax.bar(["No perm.", "PERMANENTE"], c.values, color=[DORADO, GRANATE])
    ax.set_title(f"{t}  ({c[1]/len(yy)*100:.0f}% positivos)")
    for i, v in enumerate(c.values): ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

print(f"Train original : {len(y_train):,} filas")
print(f"Train con SMOTE: {len(y_train_sm):,} filas  (se sintetizaron {len(y_train_sm)-len(y_train):,} casos permanentes)")
print(f"TEST se mantiene INTACTO: {y_test.mean()*100:.1f}% permanentes (proporción real) ✓")


---
## 4️⃣ Los 5 algoritmos

| # | Modelo | Por qué está aquí |
|---|---|---|
| 1 | **Regresión Logística** | *Baseline* lineal e interpretable |
| 2 | **Árbol de Decisión** | Reglas explícitas, no lineal |
| 3 | **Random Forest** | *Ensemble* por bagging — robusto |
| 4 | **XGBoost** | *Ensemble* por boosting — suele ganar en datos tabulares |
| 5 | **KNN** | Basado en distancia — enfoque distinto al resto |


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             precision_score, recall_score, f1_score, accuracy_score, roc_curve)
import time

modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=42),
    "Árbol de Decisión":   DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=15,
                                                  random_state=42, n_jobs=-1),
    "XGBoost":             XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                         random_state=42, eval_metric="logloss", n_jobs=-1),
    "KNN":                 KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
}

resultados, entrenados = [], {}
for nombre, modelo in modelos.items():
    t0 = time.time()
    modelo.fit(X_train_sm, y_train_sm)            # ← entrenan con SMOTE
    y_pred  = modelo.predict(X_test)              # ← evalúan en el test REAL
    y_proba = modelo.predict_proba(X_test)[:, 1]
    resultados.append({
        "Modelo": nombre,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precisión": precision_score(y_test, y_pred, zero_division=0),
        "Recall":    recall_score(y_test, y_pred),
        "F1":        f1_score(y_test, y_pred),
        "ROC-AUC":   roc_auc_score(y_test, y_proba),
        "seg":       round(time.time()-t0, 1),
    })
    entrenados[nombre] = modelo
    print(f"✓ {nombre:22s} entrenado en {time.time()-t0:5.1f}s")

tabla = pd.DataFrame(resultados).set_index("Modelo").sort_values("Recall", ascending=False)


---
## 5️⃣ Comparación de modelos

> **¿Por qué ordenamos por RECALL y no por Accuracy?**
> Con 18 % de positivos, un modelo que diga *"nada es permanente"* obtiene **82 % de accuracy**
> y es **inútil** (recall = 0). El **falso negativo** —decir "leve" a un accidente que dejará
> al trabajador discapacitado— es el error **más caro en vidas**. Por eso priorizamos **recall**.


In [ ]:
display(tabla.style
    .background_gradient(subset=["Recall","F1","ROC-AUC"], cmap="Greens")
    .format({"Accuracy":"{:.3f}","Precisión":"{:.3f}","Recall":"{:.3f}",
             "F1":"{:.3f}","ROC-AUC":"{:.3f}","seg":"{:.1f}"}))

baseline = 1 - y_test.mean()
print(f"⚠️ Baseline tonto ('nunca es permanente'): accuracy = {baseline:.3f} pero recall = 0.000")


In [ ]:
# Gráfico comparativo
met = ["Precisión","Recall","F1","ROC-AUC"]
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(tabla)); w = 0.2
for i, m in enumerate(met):
    ax.bar(x + i*w - 1.5*w, tabla[m], w, label=m)
ax.set_xticks(x); ax.set_xticklabels(tabla.index, rotation=15, ha="right")
ax.set_ylim(0, 1); ax.set_title("Comparación de los 5 algoritmos (test 2022)")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.0))
plt.tight_layout(); plt.show()


In [ ]:
# Curvas ROC
fig, ax = plt.subplots(figsize=(6, 5))
for nombre, modelo in entrenados.items():
    proba = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, lw=1.8, label=f"{nombre} (AUC={roc_auc_score(y_test, proba):.3f})")
ax.plot([0,1],[0,1],'--',color=GRIS,lw=1, label="Azar (AUC=0.5)")
ax.set_xlabel("Tasa de falsos positivos"); ax.set_ylabel("Tasa de verdaderos positivos (Recall)")
ax.set_title("Curvas ROC"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


### Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(19, 3.6))
for ax, (nombre, modelo) in zip(axes, entrenados.items()):
    cm = confusion_matrix(y_test, modelo.predict(X_test))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Reds", cbar=False, ax=ax,
                xticklabels=["No perm.","Perm."], yticklabels=["No perm.","Perm."])
    ax.set_title(nombre, fontsize=10)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real" if ax is axes[0] else "")
plt.tight_layout(); plt.show()
print("Cuadrante inferior-izquierdo = FALSOS NEGATIVOS (los más costosos: accidentes graves no detectados).")


---
## 6️⃣ SMOTE vs `class_weight` — ¿cuál funciona mejor?
Dos estrategias distintas para el mismo problema. La comparación es un plus para la exposición.


In [ ]:
comp = []
for etiqueta, Xtr, ytr, kw in [
    ("Sin nada (desbalanceado)", X_train, y_train, {}),
    ("Con SMOTE",                X_train_sm, y_train_sm, {}),
    ("Con class_weight",         X_train, y_train, {"class_weight":"balanced"}),
]:
    m = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1, **kw)
    m.fit(Xtr, ytr)
    p = m.predict(X_test)
    comp.append({"Estrategia": etiqueta,
                 "Accuracy": accuracy_score(y_test, p),
                 "Precisión": precision_score(y_test, p, zero_division=0),
                 "Recall": recall_score(y_test, p),
                 "F1": f1_score(y_test, p)})

comp = pd.DataFrame(comp).set_index("Estrategia")
display(comp.style.background_gradient(subset=["Recall"], cmap="Greens").format("{:.3f}"))
print("Observa el intercambio: SMOTE/class_weight SUBEN el recall a costa de algo de precisión.")
print("En este problema, ese intercambio VALE LA PENA (un falso negativo cuesta una discapacidad).")


---
## 7️⃣ SHAP — explicabilidad (obligatorio)
*¿Qué empuja a que un accidente deje secuela permanente?*


In [ ]:
import shap

MEJOR = tabla.index[0]          # el de mayor recall
modelo_final = entrenados[MEJOR]
print("Modelo explicado:", MEJOR)

# Si el mejor no es de árboles, usamos XGBoost para SHAP (TreeExplainer es exacto y rápido)
if MEJOR not in ("Random Forest", "XGBoost", "Árbol de Decisión"):
    modelo_shap, nombre_shap = entrenados["XGBoost"], "XGBoost"
else:
    modelo_shap, nombre_shap = modelo_final, MEJOR

muestra = X_test.sample(min(2000, len(X_test)), random_state=42)
explainer   = shap.TreeExplainer(modelo_shap)
shap_values = explainer.shap_values(muestra)
if isinstance(shap_values, list): shap_values = shap_values[1]
print("SHAP calculado sobre", len(muestra), "casos del test.")


In [ ]:
# SUMMARY PLOT (importancia global)
shap.summary_plot(shap_values, muestra, plot_type="bar", max_display=12, show=False)
plt.title(f"SHAP · variables más influyentes ({nombre_shap})", color=GRANATE, weight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# BEESWARM (dirección del efecto: rojo = valor alto de la variable)
shap.summary_plot(shap_values, muestra, max_display=12, show=False)
plt.title("SHAP · ¿en qué dirección empuja cada variable?", color=GRANATE, weight="bold")
plt.tight_layout(); plt.show()
print("Rojo a la derecha  = valores ALTOS de esa variable AUMENTAN el riesgo de secuela permanente.")


In [ ]:
# FORCE PLOT — explicación de UN caso individual (esto alimentará el Panel 4)
i = int(np.argmax(modelo_shap.predict_proba(muestra)[:, 1]))   # el caso de mayor riesgo
print(f"Caso de MAYOR riesgo predicho: {modelo_shap.predict_proba(muestra)[i,1]*100:.1f}% de probabilidad")

shap.force_plot(explainer.expected_value if np.isscalar(explainer.expected_value)
                else explainer.expected_value[1],
                shap_values[i], muestra.iloc[i], matplotlib=True, show=False)
plt.tight_layout(); plt.show()
print("Rojo = empuja hacia PERMANENTE · Azul = empuja hacia NO permanente")


---
## 8️⃣ Ajuste del umbral de decisión
Por defecto el modelo usa 0.5. Pero **el umbral es una perilla de negocio**:
bajarlo detecta más accidentes graves (↑recall) a costa de más falsas alarmas (↓precisión).


In [ ]:
proba = modelo_final.predict_proba(X_test)[:, 1]
filas = []
for u in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (proba >= u).astype(int)
    filas.append({"Umbral": u,
                  "Precisión": precision_score(y_test, p, zero_division=0),
                  "Recall": recall_score(y_test, p),
                  "F1": f1_score(y_test, p)})
um = pd.DataFrame(filas).set_index("Umbral")

fig, ax = plt.subplots(figsize=(7, 3.6))
for c, col in zip(["Precisión","Recall","F1"], [AZUL, GRANATE, VERDE]):
    ax.plot(um.index, um[c], "o-", color=col, label=c)
ax.axvline(0.5, ls="--", color=GRIS, lw=1)
ax.set_xlabel("Umbral de decisión"); ax.set_title("El intercambio precisión ↔ recall")
ax.legend(); plt.tight_layout(); plt.show()
display(um.round(3))


---
## 9️⃣ Entrenar y guardar las DOS particiones (para el dashboard)

Para poder responder en vivo la pregunta *"¿qué pasa si cambias el train/test?"*, entrenamos
**dos particiones temporales**, cada una con sus **5 modelos**, y guardamos todo:

| Partición | Train | Test |
|---|---|---|
| **P0** (principal) | 2018–2021 | 2022 |
| **P1** (experimento) | 2018–2020 | 2021 |

El dashboard cargará el `.pkl` que el usuario elija en un combobox.


In [ ]:
import joblib, os
os.makedirs("../models", exist_ok=True)

def entrenar_particion(anios_train, anio_test, nombre):
    """Entrena los 5 modelos para una partición temporal y devuelve todo lo necesario."""
    tr = df[df["ANIOS"].isin(anios_train)].copy()
    te = df[df["ANIOS"] == anio_test].copy()
    g = tr["PERMANENTE"].mean()

    # target encoding aprendido SOLO del train de esta partición
    tasas = {}
    for col, nom in [("ACTIVIDAD_ECONOMICA","TASA_SECTOR"),
                     ("REGION","TASA_REGION"),
                     ("FORMA_DEL_ACCIDENTE_G","TASA_FORMA")]:
        mapa = tr.groupby(col)["PERMANENTE"].mean()
        tr[nom] = tr[col].map(mapa)
        te[nom] = te[col].map(mapa).fillna(g)
        tasas[col] = mapa.to_dict()

    Xtr = pd.get_dummies(tr[CAT + NUM], columns=CAT, drop_first=True, dtype=int)
    Xte = pd.get_dummies(te[CAT + NUM], columns=CAT, drop_first=True, dtype=int).reindex(columns=Xtr.columns, fill_value=0)
    ytr, yte = tr["PERMANENTE"], te["PERMANENTE"]

    sc = StandardScaler().fit(Xtr[NUM])
    Xtr[NUM] = sc.transform(Xtr[NUM]); Xte[NUM] = sc.transform(Xte[NUM])
    Xs, ys = SMOTE(random_state=42, k_neighbors=5).fit_resample(Xtr, ytr)

    def nuevos_modelos():
        return {
            "Regresión Logística": LogisticRegression(max_iter=1000, random_state=42),
            "Árbol de Decisión":   DecisionTreeClassifier(max_depth=10, random_state=42),
            "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
            "XGBoost":             XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                                 random_state=42, eval_metric="logloss", n_jobs=-1),
            "KNN":                 KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
        }

    modelos_ent, metricas = {}, []
    for nom, mod in nuevos_modelos().items():
        mod.fit(Xs, ys)
        p = mod.predict(Xte); pr = mod.predict_proba(Xte)[:, 1]
        modelos_ent[nom] = mod
        metricas.append({
            "Modelo": nom,
            "Accuracy": round(accuracy_score(yte, p), 3),
            "Precisión": round(precision_score(yte, p, zero_division=0), 3),
            "Recall": round(recall_score(yte, p), 3),
            "F1": round(f1_score(yte, p), 3),
            "ROC-AUC": round(roc_auc_score(yte, pr), 3),
            "cm": confusion_matrix(yte, p).tolist(),
        })

    return {
        "particion": nombre,
        "train_anios": anios_train, "test_anio": anio_test,
        "n_train": len(tr), "n_test": len(te), "tasa_test": round(yte.mean()*100, 1),
        "modelos": modelos_ent, "scaler": sc,
        "columnas": list(Xtr.columns), "num_cols": NUM,
        "metricas": metricas, "tasas": tasas, "tasa_global": g,
    }

PARTICIONES = [
    ([2018,2019,2020,2021], 2022, "2018-2021 / test 2022"),
    ([2018,2019,2020],      2021, "2018-2020 / test 2021"),
]

for i, (atr, ate, nom) in enumerate(PARTICIONES):
    paq = entrenar_particion(atr, ate, nom)
    # compress=3 -> imprescindible: sin comprimir el .pkl pesa >100 MB y GitHub lo rechaza
    joblib.dump(paq, f"../models/modelo_p{i}.pkl", compress=3)
    mejor = max(paq["metricas"], key=lambda m: m["Recall"])
    tam = os.path.getsize(f"../models/modelo_p{i}.pkl") / 1e6
    linea = "P" + str(i) + " [" + nom + "] -> modelo_p" + str(i) + ".pkl (" + format(tam, ".1f") + " MB) | 5 modelos | mejor por recall: " + mejor["Modelo"] + " (" + str(mejor["Recall"]) + ")"
    print("OK  " + linea)

print("Listo: el dashboard cargara estos .pkl; el combobox elige particion + algoritmo.")


---
## 🧾 Justificación escrita del modelo elegido *(requisito del profe)*

**Modelo seleccionado: el de mayor RECALL** en la clase *permanente* (ver tabla comparativa).

**Criterio de elección — por qué recall y no accuracy:**
El costo de los dos errores es radicalmente asimétrico.
- Un **falso positivo** (predecir "permanente" y que no lo sea) → se inspecciona una empresa de más.
  Costo: unas horas de un fiscalizador.
- Un **falso negativo** (predecir "leve" y que el trabajador quede discapacitado de por vida) →
  **se pierde la oportunidad de prevenir una discapacidad permanente**. Costo: humano e irreversible.

Con clases desbalanceadas (18 % / 82 %), el **accuracy es engañoso**: un modelo trivial que nunca
prediga "permanente" alcanza **82 %** y tiene **recall = 0**. Por eso el criterio es
**recall**, con **F1** y **ROC-AUC** como métricas de control para no sacrificar toda la precisión.

**Validez del resultado:**
- *Split temporal* (train 2018–2021 / test 2022): el modelo nunca vio el futuro.
- *Target encoding* aprendido solo del train.
- *SMOTE* aplicado solo al train; el test conserva la proporción real.
- Variables de **consecuencia** excluidas → el modelo predice desde el **contexto laboral**,
  por lo que sirve para **PREVENIR**, no solo para describir.

### ➡️ Siguiente
**Panel 1** (clustering de regiones/sectores) · **Panel 3** (pronóstico mensual) · **Panel 4** (CRUD)
